In [3]:
import numpy as np    
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline    
from fizinfo import *

In [4]:
# labda súlyát feltételezzuk 0.3kg
m_ball = 0.3

# Közegellenállás nélküli gravitációs modell holdi gravitáció

def create_grav_F_m(m, g=1.625):
    """
    Létrehozza egy állandó tömegű test tömegfüggvényét és gravitációs erőfüggvényét.

    A modellben az egyetlen ható erő a gravitáció:
        F_g = (0, -m·g)

    Paraméterek:
        m : test tömege [kg]
        g : gravitációs gyorsulás [m/s²]

    Visszatérés:
        force_fun : erőfüggvény, amely az eredő erőt adja vissza [N]
        mass_fun  : tömegfüggvény, amely az aktuális tömeget adja vissza [kg]
    """

    def mass_fun(t):
        """
        A test tömege időben állandó.

        A `t` paraméter azért szerepel, mert a numerikus dinamika
        általános tömegfüggvényt vár, amely akár időfüggő is lehet.
        """
        return m

    def force_fun(t, r, v, m):
        """
        Gravitációs erő 2D-ben.

        A modell homogén gravitációs mezőt feltételez, ezért az erő
        nem függ az időtől, a helyzettől vagy a sebességtől.
        """

        if t < 5:
            g=1.625
        else:
            g=9.81

        return np.array([0.0, -m * g], dtype=np.float64)

    return force_fun, mass_fun


# Gravitációs modell létrehozása 0.45 kg tömegű testre
F_grav_switch, m_const_switch = create_grav_F_m(m_ball)


In [ ]:
# Egyszerű leállási feltétel: ha y<0, akkor stop
def stop_ground_y(r, v):
    """Leállás y<0 esetén"""
    
    return r[1]<0.0

# Az egyik versenyző 16 m/s kezdősebességet tud a labdának adni, és a labdát 180 cm magasan engedi el.
# --- Kezdőfeltételek ---
kezdo_x = 0.0  # kezdeti x pozíció [m]
kezdo_y = 1.8  # kezdeti y pozíció [m] (1.8 m magasról indítjuk)

indulo_sebesseg     = 16.0   # kezdeti sebesség nagysága [m/s]
indulo_szog_fok     = 45.0   # indulási szög (vízszintestől) [°]

# Sebességvektor felbontása x és y komponensre a szög alapján:
kezdo_vx = indulo_sebesseg * np.cos(np.radians(indulo_szog_fok))  # vízszintes sebesség [m/s]
kezdo_vy = indulo_sebesseg * np.sin(np.radians(indulo_szog_fok))  # függőleges sebesség [m/s]

# holdi szimuláció ------ 5> másodperc
ball_din = num_dinam(2)                     # 2D mozgás (x-y sík)
ball_din.set_time_range(0.0, 30, 0.01)    # t ∈ [0, 30] s, dt = 0,001 s
ball_din.set_F_fun(F_grav_switch)           # erőfüggvény (gravitáció + drag + tolóerő)
ball_din.set_mass_fun(m_const_switch)      # tömegfüggvény (hajtóanyag-fogyás)
ball_din.set_r0_v0([kezdo_x, kezdo_y],     # kezdeti helyzet:  [0 m, 1.8 m]
                     [kezdo_vx, kezdo_vy]) # kezdeti sebesség: 55°, 10 m/s
ball_din.set_stop_cond(stop_ground_y)   # a fenti függvény megadása

# --- Integráció futtatása ---
ball_din.full_dinam_calc()                 # numerikus integráció (pl. RK4)

# --- Pályagörbe megjelenítése ---
#ball_din.plot_rcomp()                      # x(t) és y(t) komponensek külön ábrán


In [ ]:
# A pálya végpontjának (utolsó pozíció) távolsága az origótól.
# ball_din.r[-1]        → az utolsó [x, y] pozícióvektor (1D, shape: (2,))
# [None, :]             → 2D-ssé alakítja (shape: (1,2)), mert vect_abs ezt várja
# vect_abs(...)         → kiszámítja a vektor hosszát: sqrt(x² + y²)
# [0]                   → a visszakapott 1 elemű tömbből kiveszi a skalár értéket
tavolsag = vect_abs(ball_din.r[-1][None,:])[0]
